In [1]:
import re

def normalize_section(section: str) -> str:
    if not section:
        return "default"

    section = section.lower().strip()
    section = re.sub(r"[^\w\s]", "", section)
    section = re.sub(r"\s+", " ", section)

    return section

In [2]:
import sys
from typing import Set, List

from sqlalchemy import select, func

from src.RAG.models import ChunkModel
from src.db.main import get_session
from src.Utils.logger_setup import setup_logger, current_logger, track_performance
from src.Utils.exception_handler import CustomException


logger = setup_logger("section_service")
current_logger.set(logger)


class SectionService:
    """
    Service to extract UNIQUE normalized section names
    from ChunkModel JSONB metadata.
    """

    @track_performance
    async def get_normalized_sections(self) -> List[str]:
        """
        Fetch unique section_name values from JSONB metadata,
        normalize them, and return sorted list.
        """

        try:
            sections: Set[str] = set()

            async for session in get_session():

                # ✅ DB-level DISTINCT extraction (fast)
                stmt = select(
                    func.distinct(
                        ChunkModel.chunk_metadata["section_name"].astext
                    )
                ).where(
                    ChunkModel.chunk_metadata["section_name"].isnot(None)
                )

                result = await session.execute(stmt)

                for row in result.scalars().all():
                    if not row:
                        continue

                    normalized = normalize_section(str(row).strip())
                    sections.add(normalized)

                break  # single session usage

            logger.info(f"Extracted {len(sections)} normalized sections.")
            return sorted(sections)

        except Exception as e:
            logger.error("Failed to extract normalized section names.")
            raise CustomException(e, sys)

In [3]:
sections_names = await SectionService().get_normalized_sections()

2026-04-16 22:22:16,627 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-04-16 22:22:16,628 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-16 22:22:16,630 INFO sqlalchemy.engine.Engine select current_schema()
2026-04-16 22:22:16,630 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-16 22:22:16,630 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-04-16 22:22:16,630 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-16 22:22:16,648 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-16 22:22:16,662 INFO sqlalchemy.engine.Engine SELECT distinct(document_chunks.chunk_metadata ->> $1::TEXT) AS distinct_1 
FROM document_chunks 
WHERE document_chunks.chunk_metadata[$2::TEXT] IS NOT NULL
2026-04-16 22:22:16,664 INFO sqlalchemy.engine.Engine [generated in 0.00197s] ('section_name', 'section_name')
[2026-04-16 22:22:16,693] [INFO] Extracted 7 normalized sections.
[2026-04-16 22:22:16,695] [INFO] get_normalized_sections completed in 0.22s


2026-04-16 22:22:16,708 INFO sqlalchemy.engine.Engine ROLLBACK


In [4]:
sections_names

['drivesure auto loan dal',
 'edugrow education loan',
 'flexiserve personal loan fpl',
 'homesecure home loan hhl',
 'homesure mortgage loan hml',
 'securesave loan against approved securities slaas',
 'tradeplus business loan tbl']

In [13]:
import json
import numpy as np
import ollama
import asyncio



class QueryClassifier:

    def __init__(self, llm_model="qwen2.5:7b", embed_model="nomic-embed-text", threshold=0.55):
        self.llm_model = llm_model
        self.embed_model = embed_model
        self.threshold = threshold

        self.section_service = SectionService()

        self.section_list = []
        self.section_embeddings = None

    # ----------------------------
    # SINGLE EMBEDDING CALL (OLLAMA)
    # ----------------------------
    async def _embed(self, text: str):

        def _call():
            return ollama.embeddings(
                model=self.embed_model,
                prompt=f"search_document: {text}"
            )

        response = await asyncio.to_thread(_call)
        return response["embedding"]

    # ----------------------------
    # Load + embed DB sections
    # ----------------------------
    async def prepare_context(self):

        self.section_list = await self.section_service.get_normalized_sections()

        if not self.section_list:
            self.section_list = ["default"]

        vectors = []
        for s in self.section_list:
            emb = await self._embed(s)
            vectors.append(emb)

        vectors = np.array(vectors, dtype="float32")

        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9
        self.section_embeddings = vectors / norms

    # ----------------------------
    # cosine match helper
    # ----------------------------
    async def _cosine_match(self, text: str):

        vec = await self._embed(text)
        vec = np.array(vec, dtype="float32")
        vec = vec / (np.linalg.norm(vec) + 1e-9)

        sims = np.dot(self.section_embeddings, vec)

        top2_idx = np.argsort(sims)[-2:][::-1]

        results = []
        for idx in top2_idx:
            if sims[idx] >= self.threshold:
                results.append(self.section_list[idx])

        return results[:2] if results else self.section_list[:2]

    # ----------------------------
    # MAIN: batch classification
    # ----------------------------
    async def classify_batch(self, queries: list[str]) -> dict:

        if not self.section_list:
            await self.prepare_context()

        prompt = f"""
You are a classifier.

Available sections:
{self.section_list}

Return JSON like:
{{
  "query": ["section1", "section2"]
}}

Queries:
{queries}
"""

        response = ollama.chat(
            model=self.llm_model,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0}
        )

        content = response["message"]["content"]

        try:
            raw_output = json.loads(content)

            final_output = {}

            for query, sections in raw_output.items():

                normalized_sections = []

                for s in sections:
                    s = normalize_section(s)
                    matches = await self._cosine_match(s)
                    normalized_sections.extend(matches)

                final_output[query] = list(dict.fromkeys(normalized_sections))[:2]

            return final_output

        except Exception:

            # fallback: embedding-only routing
            return {
                q: await self._cosine_match(q)
                for q in queries
            }

In [14]:
query_batch = ['detail the terms of an automobile financing plan',
 'outline the conditions for obtaining a vehicle loan']

In [15]:
section_dict = await QueryClassifier().classify_batch(query_batch)

2026-04-16 22:38:26,545 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-16 22:38:26,549 INFO sqlalchemy.engine.Engine SELECT distinct(document_chunks.chunk_metadata ->> $1::TEXT) AS distinct_1 
FROM document_chunks 
WHERE document_chunks.chunk_metadata[$2::TEXT] IS NOT NULL
2026-04-16 22:38:26,550 INFO sqlalchemy.engine.Engine [cached since 969.9s ago] ('section_name', 'section_name')
[2026-04-16 22:38:26,565] [INFO] Extracted 7 normalized sections.
[2026-04-16 22:38:26,568] [INFO] get_normalized_sections completed in 0.04s


2026-04-16 22:38:26,580 INFO sqlalchemy.engine.Engine ROLLBACK


In [16]:
section_dict

{'detail the terms of an automobile financing plan': ['drivesure auto loan dal',
  'flexiserve personal loan fpl'],
 'outline the conditions for obtaining a vehicle loan': ['drivesure auto loan dal',
  'edugrow education loan']}

In [38]:
import faiss
import numpy as np
from collections import defaultdict
from sqlalchemy import select
from src.RAG.models import ChunkModel
from src.db.main import get_session
from src.Utils.logger_setup import setup_logger, current_logger, track_performance
from src.Utils.exception_handler import CustomException


logger = setup_logger("vectorstore")
current_logger.set(logger)


class VectorStore:

    def __init__(self, dim=768):
        self.dim = dim

        # section -> FAISS index
        self.indexes = {}

        # section -> {fid -> ChunkModel}
        self.chunk_store = {}

        self.section_service = SectionService()
        self.valid_sections = set()

    # -------------------------------------------------
    # LOAD CANONICAL SECTIONS
    # -------------------------------------------------
    async def load_sections(self):

        sections = await self.section_service.get_normalized_sections()
        self.valid_sections = set(sections)

        for sec in self.valid_sections:
            if sec not in self.indexes:
                self.indexes[sec] = faiss.IndexHNSWFlat(self.dim, 32)
                self.chunk_store[sec] = {}

    # -------------------------------------------------
    # BUILD VECTOR STORE FROM DB
    # -------------------------------------------------
    @track_performance
    async def add(self):

        try:
            if not self.valid_sections:
                await self.load_sections()

            grouped = defaultdict(list)

            # ---------------- DB FETCH ----------------
            async for session in get_session():

                stmt = select(ChunkModel)
                result = await session.execute(stmt)
                chunks = result.scalars().all()

                for c in chunks:

                    # ❗ SAFE embedding check (FIXED BUG)
                    if c.embedding is None:
                        logger.warning(f"Skipping chunk {c.id} (no embedding)")
                        continue

                    embedding = np.asarray(c.embedding, dtype="float32")

                    if embedding.size == 0:
                        logger.warning(f"Skipping chunk {c.id} (empty embedding)")
                        continue

                    section = normalize_section(
                        c.chunk_metadata.get("section_name")
                    )

                    if section not in self.valid_sections:
                        section = "default"

                    grouped[section].append((c, embedding))

                break

            # -------------------------------------------------
            # BUILD FAISS INDEX PER SECTION
            # -------------------------------------------------
            for section, items in grouped.items():

                index = self.indexes.get(section)

                if index is None:
                    index = faiss.IndexHNSWFlat(self.dim, 32)
                    self.indexes[section] = index
                    self.chunk_store[section] = {}

                vectors = np.array(
                    [emb for _, emb in items],
                    dtype="float32"
                )

                faiss.normalize_L2(vectors)

                start_id = index.ntotal
                index.add(vectors)

                for i, (chunk, _) in enumerate(items):
                    fid = start_id + i
                    self.chunk_store[section][fid] = chunk

            logger.info(
                f"VectorStore built successfully | sections={len(grouped)}"
            )

        except Exception as e:
            logger.error("Failed to build VectorStore from DB")
            raise CustomException(e)

    # -------------------------------------------------
    # SEARCH
    # -------------------------------------------------
    def search(self, query_vector, sections, k=5):

        query = np.asarray(query_vector, dtype="float32").reshape(1, -1)
        faiss.normalize_L2(query)

        results = []

        for sec in sections:

            sec = normalize_section(sec)

            index = self.indexes.get(sec)
            if not index:
                continue

            D, I = index.search(query, k)

            for score, idx in zip(D[0], I[0]):

                if idx == -1:
                    continue

                chunk = self.chunk_store[sec].get(idx)

                if chunk:
                    results.append({
                        "chunk": chunk,
                        "score": float(score)
                    })

        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:k]

In [39]:
import asyncio

store = VectorStore(dim=768)

await store.add()

2026-04-16 23:16:36,065 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-16 23:16:36,082 INFO sqlalchemy.engine.Engine SELECT distinct(document_chunks.chunk_metadata ->> $1::TEXT) AS distinct_1 
FROM document_chunks 
WHERE document_chunks.chunk_metadata[$2::TEXT] IS NOT NULL
2026-04-16 23:16:36,083 INFO sqlalchemy.engine.Engine [cached since 3259s ago] ('section_name', 'section_name')
[2026-04-16 23:16:36,110] [INFO] Extracted 7 normalized sections.
[2026-04-16 23:16:36,110] [INFO] get_normalized_sections completed in 0.13s
2026-04-16 23:16:36,128 INFO sqlalchemy.engine.Engine ROLLBACK
2026-04-16 23:16:36,128 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-16 23:16:36,128 INFO sqlalchemy.engine.Engine SELECT document_chunks.id, document_chunks.chunk_text, document_chunks.embedding, document_chunks.confidence_score, document_chunks.chunk_metadata, document_chunks.prev_chunk_id, document_chunks.next_chunk_id, document_chunks.created_at, document_chunks.context_chunks 
FROM d

2026-04-16 23:16:36,745 INFO sqlalchemy.engine.Engine ROLLBACK


In [40]:
from src.RAG.models import ChunkModel
from src.RAG.Strategies_RAG_build.embeddings import DocumentEmbedder
import asyncio

async def run_flat_search(store, section_dict, k=10):

    all_results = []

    for query, sections in section_dict.items():

        # 1. embed query
        embedder = DocumentEmbedder()
        query_vector = await embedder.embed_query(query)

        # 2. search FAISS
        results = store.search(
            query_vector=query_vector,
            sections=sections,
            k=k
        )

        # 3. append in required format
        for r in results:
            all_results.append({
                "chunk": r["chunk"],   # ChunkModel
                "score": r["score"]
            })

    # optional: global sort across all queries
    all_results.sort(key=lambda x: x["score"], reverse=True)

    return all_results[:k]

In [41]:
sections_names

['drivesure auto loan dal',
 'edugrow education loan',
 'flexiserve personal loan fpl',
 'homesecure home loan hhl',
 'homesure mortgage loan hml',
 'securesave loan against approved securities slaas',
 'tradeplus business loan tbl']

In [42]:
section_dict

{'detail the terms of an automobile financing plan': ['drivesure auto loan dal',
  'flexiserve personal loan fpl'],
 'outline the conditions for obtaining a vehicle loan': ['drivesure auto loan dal',
  'edugrow education loan']}

In [46]:
results = await run_flat_search(
    store=store,
    section_dict= section_dict,
    k=10
)

print(results)

[{'chunk': ChunkModel(embedding=array([ 1.17219508e+00,  9.70847845e-01, -3.37891889e+00, -6.68012440e-01,
        1.31677449e+00, -1.09881245e-01, -2.98895091e-01, -1.75087795e-01,
        6.34573638e-01, -2.41394386e-01, -5.44358373e-01,  2.92793125e-01,
        2.21492365e-01,  1.51848435e-01, -6.79295557e-03, -1.71685740e-01,
        6.66074276e-01, -1.00670588e+00,  6.12501562e-01,  1.29740524e+00,
       -6.10397995e-01, -5.87220371e-01, -9.35743332e-01, -3.51265460e-01,
        6.62998974e-01,  3.40070993e-01,  1.00456119e+00, -1.12361681e+00,
       -4.90780711e-01, -4.18359995e-01, -5.39066017e-01, -7.03087077e-03,
        6.17860019e-01, -1.71402311e+00, -6.76068485e-01, -8.30108106e-01,
        1.62586367e+00,  5.96791327e-01,  4.56302851e-01, -2.05669150e-01,
       -1.37571943e+00,  1.53693688e+00, -6.41478077e-02, -7.48576343e-01,
        1.31345451e+00, -8.66576910e-01,  1.89658606e+00,  7.40951002e-01,
        9.42142904e-01, -3.07815939e-01,  4.43528533e-01, -2.7055278

In [44]:
for sec, idx in store.indexes.items():
    print(sec, idx.ntotal)

homesecure home loan hhl 53
drivesure auto loan dal 86
edugrow education loan 92
flexiserve personal loan fpl 78
homesure mortgage loan hml 119
tradeplus business loan tbl 113
securesave loan against approved securities slaas 73
